# Experiment: 10 Prefix Filter Analysis

Objective: evaluate the abort-and-retry prefix filtering policy across all score kinds, models, and scales.

Score kinds:
- `entropy_only` — token-level entropy baseline (no geometry)
- `mahalanobis_only` — Mahalanobis distance summary stats at a single layer (logistic classifier)
- `combined` — logistic mix of entropy + Mahalanobis features
- `mahal_raw` — negative cumulative Mahalanobis distance, no classifier (requires re-running `evaluate_prefix_filter`)
- `mahal_cumtraj` — trajectory shape features from the cumsum (requires re-running `evaluate_prefix_filter`)

Key questions:
1. Does any score kind + operating point achieve positive token savings *and* non-negative accuracy?
2. Why is false abort rate persistently ~0.5 (near random)?
3. Do `mahal_raw` / `mahal_cumtraj` improve over the logistic baseline?

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

candidate_dirs = [Path.cwd(), Path.cwd() / "notebooks", *[p / "notebooks" for p in Path.cwd().parents]]
for _c in candidate_dirs:
    if (_c / "_viz_utils.py").exists():
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break

import _viz_utils as vu

ROOT = vu.repo_root()
RESULTS = ROOT / "results"

RUNS = [
    {"model": "qwen",     "scale": "pilot", "dir": "qwen_prefix_filter_pilot"},
    {"model": "qwen",     "scale": "full",  "dir": "qwen_prefix_filter_full"},
    {"model": "deepseek", "scale": "pilot", "dir": "deepseek_prefix_filter_pilot"},
    {"model": "deepseek", "scale": "full",  "dir": "deepseek_prefix_filter_full"},
]

def load_pf(run: dict) -> dict:
    path = RESULTS / run["dir"] / "math500" / "math500_prefix_filter_results.json"
    return json.loads(path.read_text())

data = {r["dir"]: load_pf(r) for r in RUNS}
print("Loaded:", list(data.keys()))

In [ ]:
def flatten_settings(run_key: str, d: dict) -> list[dict]:
    """Flatten all settings × thresholds into rows."""
    model, scale = run_key.split("_prefix_filter_")
    rows = []
    baseline = next(iter(d["settings"].values()))["baseline"]
    for label, setting in d["settings"].items():
        for q_str, t in setting["thresholds"].items():
            if t.get("skipped"):
                continue
            rows.append({
                "run":             run_key,
                "model":           model,
                "scale":           scale,
                "score_kind":      setting["score_kind"],
                "layer":           setting["layer"],
                "prefix_len":      setting["prefix_len"],
                "quantile":        t["quantile"],
                "pass_at_1":       t["pass_at_1_mean"],
                "pass_delta":      t["pass_delta_vs_baseline"],
                "tokens":          t["expected_tokens_mean"],
                "token_savings":   t["token_savings_mean"],
                "false_abort_rate":t["false_abort_rate"],
                "avg_aborts":      t["avg_aborts_per_problem"],
                "first_try_accept":t["first_try_accept_rate"],
                "baseline_pass":   baseline["pass_at_1_mean"],
                "baseline_tokens": baseline["expected_tokens_mean"],
            })
    return rows

rows = []
for key, d in data.items():
    rows.extend(flatten_settings(key, d))

df = pd.DataFrame(rows)
print(f"{len(df)} operating points across {df['run'].nunique()} runs")
print("Score kinds present:", sorted(df['score_kind'].unique()))
df.head(3)

## Baselines

In [ ]:
baselines = (
    df.groupby(["run", "model", "scale"])
    .first()[["baseline_pass", "baseline_tokens"]]
    .reset_index()
    .rename(columns={"baseline_pass": "Pass@1 (baseline)", "baseline_tokens": "Tokens (baseline)"})
)
baselines["Tokens (baseline)"] = baselines["Tokens (baseline)"].round(1)
baselines["Pass@1 (baseline)"] = baselines["Pass@1 (baseline)"].round(3)
baselines

## Does any operating point achieve positive token savings?

Key question: is there any (score kind, layer, prefix, quantile) combo that saves tokens without losing accuracy?

In [ ]:
positive = df[(df["token_savings"] > 0)]
print(f"Operating points with positive token savings: {len(positive)} / {len(df)}")
if len(positive):
    display(positive[["run", "score_kind", "layer", "prefix_len", "quantile",
                       "pass_delta", "token_savings", "false_abort_rate"]].sort_values("token_savings", ascending=False))
else:
    print("None — no operating point yields positive token savings across any run.")

## Tradeoff landscape: token savings vs accuracy delta

Each point is one (score kind × layer × prefix × quantile) combination. The upper-right quadrant (savings > 0 AND delta > 0) is the target region.

In [ ]:
KIND_COLORS = {
    "entropy_only":      "#999999",
    "mahalanobis_only":  "#4C72B0",
    "combined":          "#DD8452",
    "mahal_raw":         "#55A868",
    "mahal_cumtraj":     "#C44E52",
}

run_labels = {
    "qwen_prefix_filter_pilot":     "Qwen pilot",
    "qwen_prefix_filter_full":      "Qwen full",
    "deepseek_prefix_filter_pilot": "DeepSeek pilot",
    "deepseek_prefix_filter_full":  "DeepSeek full",
}

fig, axes = plt.subplots(2, 2, figsize=(13, 10), sharex=False, sharey=False)
axes = axes.flat

for ax, run in zip(axes, RUNS):
    sub = df[df["run"] == run["dir"]]
    for kind, grp in sub.groupby("score_kind"):
        ax.scatter(
            grp["token_savings"], grp["pass_delta"],
            label=kind, color=KIND_COLORS.get(kind, "black"),
            s=30, alpha=0.7, edgecolors="none",
        )
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.axvline(0, color="black", lw=0.8, ls="--")
    ax.set_title(run_labels[run["dir"]], fontsize=11)
    ax.set_xlabel("Token savings (fraction)")
    ax.set_ylabel("Pass@1 delta vs baseline")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=len(KIND_COLORS), fontsize=9, title="Score kind")
fig.suptitle("Prefix filter tradeoff: token savings vs accuracy", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## False abort rate: near-random across all settings

If the scorer has no signal, aborted traces should have the same correctness rate as the full population (~baseline pass@1 ≈ 0.55–0.57 for Qwen, 0.40–0.42 for DeepSeek).  
A false abort rate *below* the base rate indicates the classifier is preferentially aborting incorrect traces.  
A rate *at or above* base rate means the classifier has essentially no discriminative signal.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes = axes.flat

for ax, run in zip(axes, RUNS):
    sub = df[df["run"] == run["dir"]]
    base_rate = sub["baseline_pass"].iloc[0]

    for kind, grp in sub.groupby("score_kind"):
        sorted_grp = grp.sort_values("quantile")
        ax.scatter(
            sorted_grp["avg_aborts"], sorted_grp["false_abort_rate"],
            label=kind, color=KIND_COLORS.get(kind, "black"),
            s=30, alpha=0.7, edgecolors="none",
        )

    ax.axhline(base_rate, color="red", lw=1.2, ls="--", label=f"base rate ({base_rate:.2f})")
    ax.axhline(0.5, color="grey", lw=0.8, ls=":")
    ax.set_title(run_labels[run["dir"]], fontsize=11)
    ax.set_xlabel("Avg aborts per problem")
    ax.set_ylabel("False abort rate")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7, ncol=2)

fig.suptitle("False abort rate vs abort aggressiveness", fontsize=13)
plt.tight_layout()
plt.show()

## Best operating point per score kind

In [ ]:
best = (
    df.sort_values("pass_delta", ascending=False)
    .groupby(["run", "score_kind"], sort=False)
    .first()
    .reset_index()
    [["run", "score_kind", "layer", "prefix_len", "quantile",
      "pass_delta", "token_savings", "false_abort_rate", "avg_aborts"]]
)
best["pass_delta"]    = best["pass_delta"].map(lambda x: f"{x:+.3f}")
best["token_savings"] = best["token_savings"].map(lambda x: f"{x:+.3f}")
best["false_abort_rate"] = best["false_abort_rate"].round(3)
best["avg_aborts"]    = best["avg_aborts"].round(2)
best

## Effect of prefix length

Longer prefixes give the scorer more signal but cost more tokens per abort.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for run_key, ax_label, ax in [
    ("qwen_prefix_filter_full",      "Qwen full",      axes[0]),
    ("deepseek_prefix_filter_full",  "DeepSeek full",  axes[1]),
]:
    sub = df[(df["run"] == run_key) & (df["score_kind"] == "mahalanobis_only")]
    for plen, grp in sub.groupby("prefix_len"):
        grp_sorted = grp.sort_values("quantile")
        ax.plot(
            grp_sorted["token_savings"], grp_sorted["pass_delta"],
            marker="o", ms=5, label=f"k={plen}",
        )
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.axvline(0, color="black", lw=0.8, ls="--")
    ax.set_title(f"{ax_label} — mahalanobis_only", fontsize=11)
    ax.set_xlabel("Token savings")
    ax.set_ylabel("Pass@1 delta")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(title="Prefix tokens")

fig.suptitle("Tradeoff curves by prefix length (mahalanobis_only, full scale)", fontsize=12)
plt.tight_layout()
plt.show()

## New score kinds: `mahal_raw` and `mahal_cumtraj`

These require re-running `dvc repro evaluate_prefix_filter` after the implementation changes to `prefix_filter.py`.  
Once results are available, the cells above will automatically pick them up (score kinds are loaded from the JSON).

**`mahal_raw`**: negative cumulative Mahalanobis distance — no classifier trained, threshold calibrated from train quantiles only.  
**`mahal_cumtraj`**: trajectory shape of the cumulative sum (total, slope, convexity, peak position) fed into the existing logistic classifier.

Below: compare false abort rate for the new kinds vs baselines, if present.

In [ ]:
new_kinds = ["mahal_raw", "mahal_cumtraj"]
df_new = df[df["score_kind"].isin(new_kinds)]

if df_new.empty:
    print("mahal_raw / mahal_cumtraj not yet in results — re-run `dvc repro evaluate_prefix_filter`.")
else:
    summary = (
        df_new
        .groupby(["run", "score_kind"])[["pass_delta", "token_savings", "false_abort_rate"]]
        .agg({"pass_delta": "max", "token_savings": "max", "false_abort_rate": "mean"})
        .reset_index()
    )
    display(summary)

## Summary

- **No operating point** across any model, scale, score kind, prefix length, or threshold achieves positive token savings.
- **False abort rate ≈ 0.5** throughout — close to the base rate of correct traces (~0.57 Qwen, ~0.42 DeepSeek), indicating the prefix scorer is barely above random. The geometric signal in the first k tokens is real but very weak.
- **Longer prefixes** shift the tradeoff further into token-costly territory without recovering accuracy — the marginal information gain from more prefix tokens is outweighed by the abort cost.
- **DeepSeek** shows the best false abort rates (as low as 0.26 on full) because its traces are much longer (~1700 tokens), giving more geometric structure per prefix token, but the high per-abort cost (80–400 tokens) still makes the policy uneconomical.
- **Next**: `mahal_raw` (no classifier) and `mahal_cumtraj` (trajectory shape) may improve signal — re-run `evaluate_prefix_filter` after implementing these score kinds.